In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

# ==========================================
# 1. CONFIGURATION & HYPERPARAMETERS
# ==========================================
DATA_DIR = r"C:\Users\ashut\BE_Project\Dataset\Generated_FAF_CNN_Dataset_Refactored_3classes"
BATCH_SIZE = 8      # Kept small because the dataset is small
EPOCHS = 40           # Number of times to loop through the data
LEARNING_RATE = 0.0005
VAL_SPLIT = 0.2       # 20% of data for validation

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ==========================================
# 2. DATA AUGMENTATION & TRANSFORMS
# ==========================================
# ResNet expects images to be 224x224 and normalized with specific ImageNet values.
# For training, we add random flips and rotations to artificially "expand" your 273 images.

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10), # Slight rotations to help generalize
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# ==========================================
# 3. LOAD AND SPLIT DATASET
# ==========================================
# We load the dataset twice to apply different transforms to train and val, 
# then use Subset to split them properly.

full_train_dataset = datasets.ImageFolder(DATA_DIR, transform=train_transforms)
full_val_dataset = datasets.ImageFolder(DATA_DIR, transform=val_transforms)

class_names = full_train_dataset.classes
num_classes = len(class_names)
print(f"Found {num_classes} classes: {class_names}")

# Create random split indices
num_samples = len(full_train_dataset)
indices = torch.randperm(num_samples).tolist()
split_idx = int((1 - VAL_SPLIT) * num_samples)

train_idx, val_idx = indices[:split_idx], indices[split_idx:]

# Create DataLoaders
train_data = Subset(full_train_dataset, train_idx)
val_data = Subset(full_val_dataset, val_idx)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training images: {len(train_data)} | Validation images: {len(val_data)}")

# ==========================================
# 4. BUILD THE TRANSFER LEARNING MODEL
# ==========================================
print("\nLoading Pre-trained ResNet18...")
# Load the pre-trained ResNet18
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# FREEZE the early layers (Feature Extractor)
for param in model.parameters():
    param.requires_grad = False

# Unfreeze and replace the final fully connected (fc) layer
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, num_classes) # Maps to your ~15 classes

model = model.to(device)

# ==========================================
# 5. LOSS FUNCTION & OPTIMIZER
# ==========================================
criterion = nn.CrossEntropyLoss()
# Only optimize the parameters of the newly added final layer
optimizer = optim.Adam(model.fc.parameters(), lr=LEARNING_RATE)

# ==========================================
# 6. TRAINING LOOP
# ==========================================
print("\nStarting Training...\n")

best_val_acc = 0.0

for epoch in range(EPOCHS):
    # --- TRAINING ---
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
    epoch_train_loss = running_loss / len(train_data)
    epoch_train_acc = 100 * correct / total
    
    # --- VALIDATION ---
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
            
    epoch_val_loss = val_loss / len(val_data)
    epoch_val_acc = 100 * val_correct / val_total
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {epoch_train_loss:.4f} Acc: {epoch_train_acc:.2f}% | Val Loss: {epoch_val_loss:.4f} Acc: {epoch_val_acc:.2f}%")
    
    # Save the best model
    if epoch_val_acc > best_val_acc:
        best_val_acc = epoch_val_acc
        torch.save(model.state_dict(), 'best_resnet_faf_classifier.pth')

print(f"\n✅ Training Complete! Best Validation Accuracy: {best_val_acc:.2f}%")
print("Best model weights saved to 'best_resnet_faf_classifier.pth'")

Using device: cuda
Found 3 classes: ['Choroiditis', 'Normal', 'Scar']
Training images: 49 | Validation images: 13

Loading Pre-trained ResNet18...

Starting Training...

Epoch 1/40 | Train Loss: 1.1330 Acc: 34.69% | Val Loss: 0.9755 Acc: 61.54%
Epoch 2/40 | Train Loss: 1.1347 Acc: 40.82% | Val Loss: 1.0083 Acc: 61.54%
Epoch 3/40 | Train Loss: 1.2077 Acc: 34.69% | Val Loss: 1.0109 Acc: 61.54%
Epoch 4/40 | Train Loss: 1.1218 Acc: 38.78% | Val Loss: 1.0526 Acc: 23.08%
Epoch 5/40 | Train Loss: 1.0686 Acc: 44.90% | Val Loss: 1.0404 Acc: 38.46%
Epoch 6/40 | Train Loss: 1.0661 Acc: 34.69% | Val Loss: 1.0281 Acc: 30.77%
Epoch 7/40 | Train Loss: 1.0963 Acc: 44.90% | Val Loss: 0.9444 Acc: 53.85%
Epoch 8/40 | Train Loss: 1.0206 Acc: 42.86% | Val Loss: 0.9657 Acc: 53.85%
Epoch 9/40 | Train Loss: 1.0146 Acc: 46.94% | Val Loss: 1.0599 Acc: 30.77%
Epoch 10/40 | Train Loss: 1.0428 Acc: 40.82% | Val Loss: 1.0381 Acc: 46.15%
Epoch 11/40 | Train Loss: 0.9767 Acc: 48.98% | Val Loss: 0.9079 Acc: 53.85%
Epo